In [ ]:
import pandas as pd
import torch
from datasets import load_dataset, Dataset
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model_dir = 'models/'
inference_data_path = '../../datasets/set1/meta2024_set1_20260213.csv.gz'
output_path_fb = 'data/meta2024_predicted_goals_bert_all.csv.gz'

inference = pd.read_csv(inference_data_path, encoding='UTF-8', keep_default_na = False, dtype = 'str')

# All fields
cols = ['ad_creative_body', 'ad_creative_link_caption', 'ad_creative_link_description', 'ad_creative_link_title', 'ocr_text', 'asr_text']
# Combine and clean up
inference['text'] = inference[cols].apply(lambda row: ' '.join(row.values.astype(str)), axis=1)
inference['text'] = inference['text'].str.strip()
inference['text'] = inference['text'].str.replace(' +', ' ', regex = True) # Remove double (and triple etc.) whitespaces inside

inference = inference[inference['text'] != ""]
inference = inference.dropna()

goals = ['DONATE', 'CONTACT', 'PURCHASE', 'GOTV', 'EVENT', 'POLL', 'GATHERINFO', 'LEARNMORE', "PRIMARY_PERSUADE"]


In [ ]:
# Tokenize data
tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")
inference_dataset = Dataset.from_pandas(inference)

def tokenize_function(examples):
    return tokenizer(examples['text'], truncation=True, max_length=512)

inference_dataset = inference_dataset.map(tokenize_function, batched=True)

# Keep only what's needed for inference
keep_cols = {'input_ids', 'attention_mask'}
inference_dataset = inference_dataset.remove_columns(
    [c for c in inference_dataset.column_names if c not in keep_cols]
)

df_output = pd.DataFrame({'ad_id': inference['ad_id'].values})

batch_size = 2000

for goal in goals:
    model_path = model_dir + goal
    model = DistilBertForSequenceClassification.from_pretrained(model_path).to(device)
    model.eval()

    predicted_labels = []

    for batch_start in range(0, len(inference_dataset), batch_size):
        batch = inference_dataset[batch_start:batch_start + batch_size]

        # Pad the batch dynamically to the longest sequence in this batch
        encoded = tokenizer.pad(
            {"input_ids": batch["input_ids"], "attention_mask": batch["attention_mask"]},
            padding=True,
            return_tensors="pt",
        )
        input_ids = encoded["input_ids"].to(device)
        attention_mask = encoded["attention_mask"].to(device)

        with torch.no_grad():
            logits = model(input_ids, attention_mask=attention_mask).logits

        predicted_labels.extend(torch.argmax(logits, dim=1).tolist())
        print(f"Finished batch {batch_start}–{batch_start + batch_size} for goal: {goal}")

    df_output[goal] = predicted_labels

# Save intermediate after all goals
df_output.to_csv(output_path_fb, index=False)

goal_names = {
    "DONATE": "Donate", "CONTACT": "Contact", "PURCHASE": "Purchase",
    "GOTV": "Vote", "EVENT": "Event", "POLL": "Poll",
    "GATHERINFO": "Acquisition", "LEARNMORE": "Learn", "PRIMARY_PERSUADE": "Persuade"
}
df_output = df_output.rename(columns=goal_names)
df_output.to_csv(output_path_fb, index=False)
df_output.to_csv('data/meta2024_predicted_goals_bert_all.csv', index=False)

In [ ]:
# Google
inference_data_path = '../../datasets/set1/google2024_set1_20260213.csv.gz'
output_path_gg = 'data/google2024_predicted_goals_bert_all.csv.gz'

inference = pd.read_csv(inference_data_path, encoding='UTF-8', keep_default_na = False, dtype = 'str')

# All fields
cols = ['ad_text', 'ocr_text', 'asr_text']
# Combine and clean up
inference['text'] = inference[cols].apply(lambda row: ' '.join(row.values.astype(str)), axis=1)
inference['text'] = inference['text'].str.strip()
inference['text'] = inference['text'].str.replace(' +', ' ', regex = True) # Remove double (and triple etc.) whitespaces inside

inference = inference[inference['text'] != ""]
inference = inference.dropna()

goals = ['DONATE', 'CONTACT', 'PURCHASE', 'GOTV', 'EVENT', 'POLL', 'GATHERINFO', 'LEARNMORE', "PRIMARY_PERSUADE"]

In [ ]:
# Tokenize data
tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")
inference_dataset = Dataset.from_pandas(inference)

def tokenize_function(examples):
    return tokenizer(examples['text'], truncation=True, max_length=512)

inference_dataset = inference_dataset.map(tokenize_function, batched=True)

# Keep only what's needed for inference
keep_cols = {'input_ids', 'attention_mask'}
inference_dataset = inference_dataset.remove_columns(
    [c for c in inference_dataset.column_names if c not in keep_cols]
)

df_output = pd.DataFrame({'ad_id': inference['ad_id'].values})

batch_size = 2000

for goal in goals:
    model_path = model_dir + goal
    model = DistilBertForSequenceClassification.from_pretrained(model_path).to(device)
    model.eval()

    predicted_labels = []

    for batch_start in range(0, len(inference_dataset), batch_size):
        batch = inference_dataset[batch_start:batch_start + batch_size]

        # Pad the batch dynamically to the longest sequence in this batch
        encoded = tokenizer.pad(
            {"input_ids": batch["input_ids"], "attention_mask": batch["attention_mask"]},
            padding=True,
            return_tensors="pt",
        )
        input_ids = encoded["input_ids"].to(device)
        attention_mask = encoded["attention_mask"].to(device)

        with torch.no_grad():
            logits = model(input_ids, attention_mask=attention_mask).logits

        predicted_labels.extend(torch.argmax(logits, dim=1).tolist())
        print(f"Finished batch {batch_start}–{batch_start + batch_size} for goal: {goal}")

    df_output[goal] = predicted_labels

# Save intermediate after all goals
df_output.to_csv(output_path_gg, index=False)

goal_names = {
    "DONATE": "Donate", "CONTACT": "Contact", "PURCHASE": "Purchase",
    "GOTV": "Vote", "EVENT": "Event", "POLL": "Poll",
    "GATHERINFO": "Acquisition", "LEARNMORE": "Learn", "PRIMARY_PERSUADE": "Persuade"
}
df_output = df_output.rename(columns=goal_names)
df_output.to_csv(output_path_gg, index=False)

In [ ]:
# TV
inference_data_path = "../../datasets/entity_linking_2024/2024_tv_asr_ocr.csv"
output_path_tv = 'data/tv2024_predicted_goals_bert_all.csv.gz'

inference = pd.read_csv(inference_data_path, encoding='UTF-8', keep_default_na = False, dtype = 'str')

# All fields
cols = ['ocr_text', 'asr_text']
# Combine and clean up
inference['text'] = inference[cols].apply(lambda row: ' '.join(row.values.astype(str)), axis=1)
inference['text'] = inference['text'].str.strip()
inference['text'] = inference['text'].str.replace(' +', ' ', regex = True) # Remove double (and triple etc.) whitespaces inside

inference = inference[inference['text'] != ""]
inference = inference.dropna()

goals = ['DONATE', 'CONTACT', 'PURCHASE', 'GOTV', 'EVENT', 'POLL', 'GATHERINFO', 'LEARNMORE', "PRIMARY_PERSUADE"]


In [ ]:
# Tokenize data
tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")
inference_dataset = Dataset.from_pandas(inference)

def tokenize_function(examples):
    return tokenizer(examples['text'], truncation=True, max_length=512)

inference_dataset = inference_dataset.map(tokenize_function, batched=True)

# Keep only what's needed for inference
keep_cols = {'input_ids', 'attention_mask'}
inference_dataset = inference_dataset.remove_columns(
    [c for c in inference_dataset.column_names if c not in keep_cols]
)

df_output = pd.DataFrame({'alt': inference['alt'].values})

batch_size = 2000

for goal in goals:
    model_path = model_dir + goal
    model = DistilBertForSequenceClassification.from_pretrained(model_path).to(device)
    model.eval()

    predicted_labels = []

    for batch_start in range(0, len(inference_dataset), batch_size):
        batch = inference_dataset[batch_start:batch_start + batch_size]

        # Pad the batch dynamically to the longest sequence in this batch
        encoded = tokenizer.pad(
            {"input_ids": batch["input_ids"], "attention_mask": batch["attention_mask"]},
            padding=True,
            return_tensors="pt",
        )
        input_ids = encoded["input_ids"].to(device)
        attention_mask = encoded["attention_mask"].to(device)

        with torch.no_grad():
            logits = model(input_ids, attention_mask=attention_mask).logits

        predicted_labels.extend(torch.argmax(logits, dim=1).tolist())
        print(f"Finished batch {batch_start}–{batch_start + batch_size} for goal: {goal}")

    df_output[goal] = predicted_labels

# Save intermediate after all goals
df_output.to_csv(output_path_tv, index=False)

goal_names = {
    "DONATE": "Donate", "CONTACT": "Contact", "PURCHASE": "Purchase",
    "GOTV": "Vote", "EVENT": "Event", "POLL": "Poll",
    "GATHERINFO": "Acquisition", "LEARNMORE": "Learn", "PRIMARY_PERSUADE": "Persuade"
}
df_output = df_output.rename(columns=goal_names)
df_output.to_csv(output_path_tv, index=False)